# Module 15: Hierarchical Active Inference and Temporal Abstraction

**Spinning Up in Active Inference** | Notebook 15 of 16

---

The brain does not process the world at a single timescale. Cortical areas form a hierarchy where higher levels represent slower, more abstract dynamics and lower levels handle fast, concrete sensorimotor processing. A key computational principle: **higher levels provide contextual priors that modulate lower-level inference**.

In Active Inference, this maps onto **hierarchical generative models**:
- **Level 1 (fast):** navigation states, motor actions, moment-to-moment decisions
- **Level 2 (slow):** context states (explore vs. exploit), strategy, task identity
- **Level 3 (slowest):** goals, identity, long-term preferences

This module covers:
1. Building a 2-level hierarchical model with `HierarchicalLevel`
2. Context-dependent A matrices: how higher levels modulate perception
3. Hierarchical inference: bottom-up evidence meets top-down expectations
4. Hierarchical EFE: cross-level information gain via mutual information
5. A context-dependent T-maze where the reward location depends on hidden context

**Key references:**
- Friston, Rosch, Parr, Price & Bowman (2017). *Deep Temporal Models and Active Inference.*
- Pezzulo, Rigoli & Friston (2018). *Hierarchical Active Inference.*

**Prerequisites:** Modules 1-14.
**Time:** ~60 minutes.

In [ ]:
# -- Setup ------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from pgmax.aif.hierarchical import (
    HierarchicalLevel,
    HierarchicalGenerativeModel,
    hierarchical_infer,
    hierarchical_efe,
    evaluate_all_policies_hierarchical,
    HierarchicalAgent,
)

import sys; sys.path.insert(0, '..')
from utils.plotting import COLORS

np.random.seed(42)
print("Hierarchical AIF module loaded.")

## 15.1 Building a 2-Level Hierarchical Model

We start with the simplest possible hierarchy:

- **Level 2 (context, slow):** 2 states -- "context A" vs. "context B". These represent a hidden variable that changes slowly (or not at all within a trial).
- **Level 1 (navigation, fast):** 4 location states with 4 actions. The key: the **A-matrix at Level 1 depends on the context** at Level 2.

This models situations like:
- A T-maze where context determines which arm is rewarded
- A navigation task where the goal location changes with the season
- A social interaction where the other person's mood affects how they respond

In [ ]:
# -- Level 1: Navigation (fast timescale) -----------------------------------
# 4 states: [center, cue, left_arm, right_arm]
# 4 actions: [stay, go_cue, go_left, go_right]
# 5 observations: [center_obs, cue_A, cue_B, reward, no_reward]
#
# CONTEXT-DEPENDENT A matrix:
#   - In context A: left_arm -> reward, right_arm -> no_reward
#   - In context B: left_arm -> no_reward, right_arm -> reward
#   - The cue location reveals the context!

nav_state_names = ['center', 'cue', 'left_arm', 'right_arm']
nav_obs_names = ['center_obs', 'cue_A', 'cue_B', 'reward', 'no_reward']
nav_action_names = ['stay', 'go_cue', 'go_left', 'go_right']

num_nav_states = 4
num_nav_obs = 5
num_nav_actions = 4
num_contexts = 2  # context A, context B

# A-matrix: shape (num_contexts, num_obs, num_states)
# A[context, obs, state] = P(obs | state, context)
A_nav = np.zeros((num_contexts, num_nav_obs, num_nav_states))

# Context A: left_arm = reward, right_arm = no_reward
A_nav[0, 0, 0] = 1.0   # center -> center_obs (both contexts)
A_nav[0, 1, 1] = 1.0   # cue -> cue_A (reveals context A)
A_nav[0, 3, 2] = 1.0   # left_arm -> reward in context A
A_nav[0, 4, 3] = 1.0   # right_arm -> no_reward in context A

# Context B: left_arm = no_reward, right_arm = reward
A_nav[1, 0, 0] = 1.0   # center -> center_obs (both contexts)
A_nav[1, 2, 1] = 1.0   # cue -> cue_B (reveals context B)
A_nav[1, 4, 2] = 1.0   # left_arm -> no_reward in context B
A_nav[1, 3, 3] = 1.0   # right_arm -> reward in context B

# B-matrix: transitions, shape (num_states, num_states, num_actions)
B_nav = np.zeros((num_nav_states, num_nav_states, num_nav_actions))
# stay: self-loop
B_nav[:, :, 0] = np.eye(num_nav_states)
# go_cue: from center to cue
B_nav[1, 0, 1] = 1.0; B_nav[1, 1, 1] = 1.0
B_nav[2, 2, 1] = 1.0; B_nav[3, 3, 1] = 1.0
# go_left: from center/cue to left_arm
B_nav[2, 0, 2] = 1.0; B_nav[2, 1, 2] = 1.0
B_nav[2, 2, 2] = 1.0; B_nav[3, 3, 2] = 1.0
# go_right: from center/cue to right_arm
B_nav[3, 0, 3] = 1.0; B_nav[3, 1, 3] = 1.0
B_nav[2, 2, 3] = 1.0; B_nav[3, 3, 3] = 1.0

# C: preferences over observations
C_nav = np.array([0.0, 0.0, 0.0, 3.0, -3.0])  # prefer reward, avoid no_reward

# D: start at center
D_nav = np.array([1.0, 0.0, 0.0, 0.0])

level1 = HierarchicalLevel(
    A=A_nav,
    B=B_nav,
    C=C_nav,
    D=D_nav,
    temporal_scale=1,
    level_name='navigation',
)

print(f"Level 1 (navigation):")
print(f"  States: {num_nav_states} ({', '.join(nav_state_names)})")
print(f"  Observations: {num_nav_obs} ({', '.join(nav_obs_names)})")
print(f"  Actions: {num_nav_actions} ({', '.join(nav_action_names)})")
print(f"  Context-dependent: {level1.context_dependent}")
print(f"  Parent states expected: {level1.num_parent_states}")

In [ ]:
# -- Level 2: Context (slow timescale) --------------------------------------
# 2 states: [context_A, context_B]
# 2 observations: [context_stable, context_switch] (abstract observations)
# 1 action: [wait] (context evolves passively)
#
# The context is mostly stable (stays the same) but can occasionally switch.

ctx_state_names = ['context_A', 'context_B']
ctx_obs_names = ['stable', 'switch']

# A-matrix for context level (NOT context-dependent -- top level has no parent)
A_ctx = np.array([
    [0.9, 0.1],  # stable observation: likely when context matches expectation
    [0.1, 0.9],  # switch observation: likely when context surprises
])

# B-matrix: context is mostly stable
B_ctx = np.array([
    [[0.95], [0.05]],  # context_A -> stays A with 95% prob
    [[0.05], [0.95]],  # context_B -> stays B with 95% prob
])

# C: slight preference for stability
C_ctx = np.array([0.5, -0.5])

# D: uncertain about context initially (the whole point!)
D_ctx = np.array([0.5, 0.5])

level2 = HierarchicalLevel(
    A=A_ctx,
    B=B_ctx,
    C=C_ctx,
    D=D_ctx,
    temporal_scale=4,  # one context step = 4 navigation steps
    level_name='context',
)

print(f"Level 2 (context):")
print(f"  States: 2 ({', '.join(ctx_state_names)})")
print(f"  Temporal scale: {level2.temporal_scale} (1 context step = 4 nav steps)")
print(f"  Context-dependent: {level2.context_dependent}")

In [ ]:
# -- Assemble the hierarchy -------------------------------------------------

hierarchy = HierarchicalGenerativeModel(levels=[level1, level2])

print(f"Hierarchical model:")
print(f"  Number of levels: {hierarchy.num_levels}")
for i, level in enumerate(hierarchy.levels):
    print(f"  Level {i} ({level.level_name}): "
          f"{level.num_states} states, {level.num_obs} obs, "
          f"{level.num_actions} actions, "
          f"temporal_scale={level.temporal_scale}")

## 15.2 Context-Dependent A Matrix

The core mechanism of hierarchical AIF: the likelihood mapping at the lower level *depends on* the higher-level state. When context is uncertain, the effective A-matrix is a mixture:

$$A_{\text{eff}}(o|s) = \sum_c Q(c) \cdot A(o|s,c)$$

As context beliefs sharpen, the effective A-matrix approaches one of the two context-specific mappings. This is how top-down expectations shape perception.

In [ ]:
# -- Visualize the context-dependent A-matrix -------------------------------

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Context A
im0 = axes[0].imshow(level1.get_A_for_context(0), cmap='Blues', aspect='auto',
                      vmin=0, vmax=1)
axes[0].set_title('A-matrix under Context A\n(left=reward, right=no_reward)')
axes[0].set_xlabel('State')
axes[0].set_ylabel('Observation')
axes[0].set_xticks(range(num_nav_states))
axes[0].set_xticklabels(nav_state_names, rotation=45, ha='right')
axes[0].set_yticks(range(num_nav_obs))
axes[0].set_yticklabels(nav_obs_names)
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Context B
im1 = axes[1].imshow(level1.get_A_for_context(1), cmap='Reds', aspect='auto',
                      vmin=0, vmax=1)
axes[1].set_title('A-matrix under Context B\n(left=no_reward, right=reward)')
axes[1].set_xlabel('State')
axes[1].set_xticks(range(num_nav_states))
axes[1].set_xticklabels(nav_state_names, rotation=45, ha='right')
axes[1].set_yticks(range(num_nav_obs))
axes[1].set_yticklabels(nav_obs_names)
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# Marginalized (uncertain context)
A_marg = level1.get_A_marginalized(np.array([0.5, 0.5]))
im2 = axes[2].imshow(A_marg, cmap='Purples', aspect='auto', vmin=0, vmax=1)
axes[2].set_title('Marginalized A-matrix\n(50/50 context uncertainty)')
axes[2].set_xlabel('State')
axes[2].set_xticks(range(num_nav_states))
axes[2].set_xticklabels(nav_state_names, rotation=45, ha='right')
axes[2].set_yticks(range(num_nav_obs))
axes[2].set_yticklabels(nav_obs_names)
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.suptitle('Context-Dependent Likelihood: Higher Levels Modulate Perception',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: under uncertainty (right panel), the arms give ambiguous")
print("observations (50% reward, 50% no_reward). Resolving context uncertainty")
print("via the cue is epistemically valuable -- it sharpens the A-matrix.")

## 15.3 Hierarchical Inference: Bottom-Up Meets Top-Down

The `hierarchical_infer` function runs alternating sweeps:

1. **Bottom-up:** Observations at each level update beliefs at that level
2. **Top-down:** Updated higher-level beliefs refine the effective A-matrix at lower levels

This alternation converges to a consistent set of beliefs across levels. The cue observation at Level 1 propagates *upward* to update context beliefs at Level 2, which then propagates *downward* to sharpen the arm observations.

In [ ]:
# -- Run hierarchical inference with different observations ------------------

# Scenario 1: Agent observes center_obs (uninformative about context)
beliefs_center = hierarchical_infer(
    hierarchy,
    observations=[0, None],  # center_obs at level 0, nothing at level 1
    num_sweeps=5,
)

print("After observing 'center_obs' (uninformative):")
print(f"  Level 0 (nav):     {beliefs_center[0].round(3)}")
print(f"  Level 1 (context): {beliefs_center[1].round(3)}")
print(f"  Context uncertain: Q(A)={beliefs_center[1][0]:.3f}, Q(B)={beliefs_center[1][1]:.3f}")

# Scenario 2: Agent observes cue_A (informative -- reveals context A!)
beliefs_cue_a = hierarchical_infer(
    hierarchy,
    observations=[1, None],  # cue_A at level 0
    num_sweeps=5,
)

print("\nAfter observing 'cue_A' (informative!):")
print(f"  Level 0 (nav):     {beliefs_cue_a[0].round(3)}")
print(f"  Level 1 (context): {beliefs_cue_a[1].round(3)}")
print(f"  Context resolved:  Q(A)={beliefs_cue_a[1][0]:.3f}, Q(B)={beliefs_cue_a[1][1]:.3f}")

# Scenario 3: Agent observes cue_B
beliefs_cue_b = hierarchical_infer(
    hierarchy,
    observations=[2, None],  # cue_B at level 0
    num_sweeps=5,
)

print("\nAfter observing 'cue_B' (informative!):")
print(f"  Level 0 (nav):     {beliefs_cue_b[0].round(3)}")
print(f"  Level 1 (context): {beliefs_cue_b[1].round(3)}")
print(f"  Context resolved:  Q(A)={beliefs_cue_b[1][0]:.3f}, Q(B)={beliefs_cue_b[1][1]:.3f}")

In [ ]:
# -- Visualize belief evolution at both levels ------------------------------

# Simulate a sequence: center -> cue -> arm
observation_sequence = [
    [0, None],  # Step 0: observe center_obs
    [1, None],  # Step 1: observe cue_A (reveals context!)
    [3, None],  # Step 2: observe reward (consistent with context A)
]

belief_trajectory = []
current_beliefs = None

for step, obs in enumerate(observation_sequence):
    current_beliefs = hierarchical_infer(
        hierarchy, obs, beliefs=current_beliefs, num_sweeps=5
    )
    belief_trajectory.append([b.copy() for b in current_beliefs])

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

steps = range(len(observation_sequence))
obs_labels = ['center_obs', 'cue_A', 'reward']

# Level 0: navigation beliefs
for s_idx, s_name in enumerate(nav_state_names):
    vals = [belief_trajectory[t][0][s_idx] for t in steps]
    axes[0].plot(steps, vals, 'o-', linewidth=2, markersize=8, label=s_name)
axes[0].set_ylabel('Belief probability')
axes[0].set_title('Level 0 (Navigation): Fast timescale beliefs')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(alpha=0.3)

# Level 1: context beliefs
for c_idx, c_name in enumerate(ctx_state_names):
    vals = [belief_trajectory[t][1][c_idx] for t in steps]
    axes[1].plot(steps, vals, 'o-', linewidth=2, markersize=8, label=c_name)
axes[1].set_ylabel('Belief probability')
axes[1].set_xlabel('Step')
axes[1].set_title('Level 1 (Context): Slow timescale beliefs')
axes[1].legend(loc='center right', fontsize=9)
axes[1].set_ylim(-0.05, 1.05)
axes[1].set_xticks(list(steps))
axes[1].set_xticklabels(obs_labels)
axes[1].grid(alpha=0.3)

# Add annotations
axes[1].annotate('Context uncertain', xy=(0, 0.5), fontsize=10, color='gray',
                ha='center', va='bottom')
axes[1].annotate('Cue reveals\ncontext A!', xy=(1, 0.95), fontsize=10,
                color='green', ha='center', va='top')

plt.suptitle('Hierarchical Belief Evolution: Bottom-Up Evidence Propagates Upward',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 15.4 Hierarchical EFE: Cross-Level Information Gain

The `hierarchical_efe` function computes the Expected Free Energy across all levels of the hierarchy. The key innovation: the **epistemic term at higher levels captures the mutual information between context and lower-level observations**.

$$G_{\text{total}} = \sum_i w_i \cdot G_i$$

where:
- $G_0$ is the standard EFE at the navigation level (pragmatic + within-level epistemic)
- $G_1$ includes a **cross-level epistemic** term: $I(o_{\text{lower}}; c) = H(o) - H(o|c)$

This mutual information term captures: "how much does knowing the context reduce uncertainty about what I will observe?" Going to the cue location produces high mutual information because the cue observation is highly informative about context.

In [ ]:
# -- Hierarchical EFE for each action ---------------------------------------

# Current beliefs: uncertain about everything
beliefs_uncertain = [D_nav.copy(), D_ctx.copy()]

G_hier = evaluate_all_policies_hierarchical(
    hierarchy, beliefs_uncertain
)

print("Hierarchical EFE for each Level-0 action:")
print(f"{'Action':<12} {'G_total':>10} {'Rank':>6}")
print("-" * 30)
ranking = np.argsort(G_hier)
for rank, idx in enumerate(ranking):
    print(f"{nav_action_names[idx]:<12} {G_hier[idx]:>10.4f} {rank+1:>6}")

print(f"\nBest action: {nav_action_names[ranking[0]]} (lowest EFE)")
print("\nThe hierarchy values 'go_cue' highly because visiting the cue")
print("provides information that reduces context uncertainty, which in turn")
print("sharpens the A-matrix at the navigation level.")

In [ ]:
# -- Compare: hierarchical EFE with different level weights -----------------

weight_configs = [
    ([1.0, 0.0], "Level 0 only (no hierarchy)"),
    ([1.0, 0.5], "Balanced hierarchy"),
    ([1.0, 1.0], "Equal weights"),
    ([1.0, 2.0], "Context-focused (high w_1)"),
]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(num_nav_actions)
width = 0.2

for i, (weights, label) in enumerate(weight_configs):
    G_w = evaluate_all_policies_hierarchical(
        hierarchy, beliefs_uncertain, level_weights=weights
    )
    ax.bar(x + i * width, G_w, width, label=f"{label}")

ax.set_xlabel('Action')
ax.set_ylabel('Expected Free Energy G (lower = better)')
ax.set_title('Effect of Level Weights on Hierarchical EFE')
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(nav_action_names)
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("With w_1 = 0 (no hierarchy), the agent does not value the cue --")
print("it treats the arms as equally likely to reward.")
print("As w_1 increases, the cue becomes more valuable because of")
print("cross-level information gain (mutual information between obs and context).")

## 15.5 Context-Dependent T-Maze: Full Agent Loop

Now we use the `HierarchicalAgent` to run a full trial of the context-dependent T-maze. The agent should:
1. Start at center, uncertain about context
2. Go to the cue location to resolve context uncertainty
3. Based on the cue, go to the correct arm for reward

In [ ]:
# -- Run a context-dependent T-maze trial -----------------------------------

# True context: A (left=reward)
true_context = 0

agent = HierarchicalAgent(
    hierarchy,
    gamma=4.0,
    level_weights=[1.0, 1.0],
    seed=42,
)

print(f"True context: {ctx_state_names[true_context]}")
print(f"Expected behavior: go_cue -> observe cue_A -> go_left -> reward")
print("=" * 60)

# Simulate: agent starts at center
current_nav_state = 0  # center

for step in range(3):
    # Generate observation from true context and current state
    A_true = level1.get_A_for_context(true_context)
    obs_probs = A_true[:, current_nav_state]
    obs = np.random.choice(num_nav_obs, p=obs_probs)

    # Agent step
    action, info = agent.step(
        observations=[obs, None],
        num_sweeps=5,
    )

    print(f"\nStep {step}:")
    print(f"  State:       {nav_state_names[current_nav_state]}")
    print(f"  Observation: {nav_obs_names[obs]}")
    print(f"  Beliefs L0:  {info['beliefs'][0].round(3)}")
    print(f"  Beliefs L1:  {info['beliefs'][1].round(3)} "
          f"(Q(A)={info['beliefs'][1][0]:.3f})")
    print(f"  EFE:         {info['G'].round(3)}")
    print(f"  Policy:      {info['policy_probs'].round(3)}")
    print(f"  Action:      {nav_action_names[action]}")

    # Transition
    transition_probs = B_nav[:, current_nav_state, action]
    current_nav_state = np.random.choice(num_nav_states, p=transition_probs)

print(f"\nFinal state: {nav_state_names[current_nav_state]}")

In [ ]:
# -- Run multiple trials and measure cue-visit rate -------------------------

n_trials = 50
cue_visit_count = 0
reward_count = 0

for trial in range(n_trials):
    true_ctx = np.random.choice(2)  # random context each trial
    agent.reset()
    nav_state = 0  # start at center
    visited_cue = False
    got_reward = False

    for step in range(3):
        A_true = level1.get_A_for_context(true_ctx)
        obs_probs = A_true[:, nav_state]
        obs = np.random.choice(num_nav_obs, p=obs_probs)

        action, _ = agent.step([obs, None], num_sweeps=3)

        transition_probs = B_nav[:, nav_state, action]
        nav_state = np.random.choice(num_nav_states, p=transition_probs)

        if nav_state == 1:  # cue location
            visited_cue = True
        if obs == 3:  # reward observation
            got_reward = True

    cue_visit_count += visited_cue
    reward_count += got_reward

print(f"Results over {n_trials} trials:")
print(f"  Cue visit rate: {cue_visit_count/n_trials*100:.1f}%")
print(f"  Reward rate:    {reward_count/n_trials*100:.1f}%")
print(f"\nA hierarchical agent that values cross-level information gain")
print(f"should visit the cue frequently, leading to higher reward rates.")

## 15.6 Temporal Abstraction: Different Timescales

In our hierarchy, Level 2 has `temporal_scale=4`, meaning one context step spans four navigation steps. This models the real-world observation that contexts change slowly while actions are fast.

This connects to the **options framework** in RL (Sutton, Precup & Singh, 1999), where temporally extended actions (options) operate at a higher level than primitive actions. The key difference: in hierarchical AIF, the timescale separation is built into the generative model, not bolted on.

In [ ]:
# -- Visualize temporal abstraction -----------------------------------------

# Simulate a longer sequence to show different timescales
n_steps = 12  # 3 context steps x 4 nav steps each

agent.reset()
nav_state = 0
true_ctx = 0  # start with context A

ctx_belief_history = []
nav_belief_history = []

for step in range(n_steps):
    # Context may switch at steps 4 and 8 (every temporal_scale steps)
    if step == 4:
        true_ctx = 1  # switch to context B
    if step == 8:
        true_ctx = 0  # switch back to context A

    A_true = level1.get_A_for_context(true_ctx)
    obs_probs = A_true[:, nav_state]
    obs = np.random.choice(num_nav_obs, p=obs_probs)

    action, info = agent.step([obs, None], num_sweeps=3)

    ctx_belief_history.append(info['beliefs'][1].copy())
    nav_belief_history.append(info['beliefs'][0].copy())

    # Transition and occasionally reset to center
    transition_probs = B_nav[:, nav_state, action]
    nav_state = np.random.choice(num_nav_states, p=transition_probs)
    if step % 4 == 3:  # reset to center every 4 steps (new mini-trial)
        nav_state = 0

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

steps_arr = np.arange(n_steps)

# Navigation beliefs (fast)
for s_idx, s_name in enumerate(nav_state_names):
    vals = [nav_belief_history[t][s_idx] for t in range(n_steps)]
    ax1.plot(steps_arr, vals, '-', linewidth=1.5, alpha=0.8, label=s_name)
ax1.set_ylabel('Belief')
ax1.set_title('Level 0 (Navigation): Fast Dynamics')
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(alpha=0.3)

# Context beliefs (slow)
for c_idx, c_name in enumerate(ctx_state_names):
    vals = [ctx_belief_history[t][c_idx] for t in range(n_steps)]
    ax2.plot(steps_arr, vals, 'o-', linewidth=2, markersize=6, label=c_name)

# Mark context switches
ax2.axvline(4, color='red', linestyle='--', alpha=0.5, label='Context switch')
ax2.axvline(8, color='red', linestyle='--', alpha=0.5)
ax2.set_ylabel('Belief')
ax2.set_xlabel('Step')
ax2.set_title('Level 1 (Context): Slow Dynamics')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(alpha=0.3)

plt.suptitle('Temporal Abstraction: Two Timescales of Belief Updating',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Navigation beliefs oscillate rapidly (step-by-step state changes).")
print("Context beliefs evolve slowly, tracking the hidden context state.")
print("After a context switch, context beliefs gradually adjust -- the")
print("agent needs evidence (cue observations) to detect the change.")

## 15.7 Exercise: Add a Third Level

**Challenge:** Extend the hierarchy to three levels:

- **Level 3 (slowest):** "task type" -- 2 states ("foraging" vs. "avoiding")
  - In "foraging" mode, the agent seeks reward
  - In "avoiding" mode, the agent avoids punishment
  - This modulates Level 2's C-vector (what the agent prefers)

- **Level 2 (medium):** context (same as before, but now context-dependent on Level 3)

- **Level 1 (fast):** navigation (same as before)

Hints:
1. Make Level 2's A-matrix 3D: `A_ctx_new = np.zeros((num_task_types, num_ctx_obs, num_ctx_states))`
2. The task type changes very slowly: `temporal_scale=16`
3. Build with `HierarchicalGenerativeModel(levels=[level1, level2_new, level3])`
4. You will need 3 entries in the `observations` list and `beliefs` list

In [ ]:
# -- Exercise skeleton: 3-level hierarchy -----------------------------------

# Level 3: Task type (very slow)
# TODO: define A, B, C, D for the task-type level
# Hint: A should be (num_obs, num_states) since this is the top level
#       and is NOT context-dependent.

# Level 2 (modified): Context, now context-dependent on task type
# TODO: make A_ctx 3D: (num_task_types, num_ctx_obs, num_ctx_states)

# Level 1: Navigation (unchanged)
# Already defined above.

print("Exercise: Create a 3-level hierarchy where:")
print("  Level 3 ('task_type'): foraging vs. avoiding")
print("  Level 2 ('context'):   context A vs. B, modulated by task type")
print("  Level 1 ('navigation'): 4 locations, modulated by context")
print("\nThe agent must infer BOTH the task type AND the context")
print("to navigate optimally. This requires deep temporal abstraction.")

## 15.8 Summary

Hierarchical Active Inference extends the single-level framework in three key ways:

1. **Context-dependent likelihood:** Higher levels modulate the A-matrix at lower levels, implementing top-down expectations that shape perception.

2. **Cross-level information gain:** The epistemic term at higher levels captures mutual information between context and observations -- the value of contextual knowledge.

3. **Temporal abstraction:** Different levels operate at different timescales, naturally separating fast sensorimotor processing from slow strategic reasoning.

These principles mirror cortical hierarchy in the brain (Friston, 2017) and provide a principled framework for multi-timescale decision making.

**Next:** [Module 16: Multi-Agent Commons](16_multiagent_commons.ipynb) -- the capstone module, where we apply everything to a multi-agent social simulation.